In [852]:
import cv2
import numpy as np

In [853]:
imagen = cv2.imread("img/img15.jpg")
gris = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)

cv2.imshow("Imagen en gris", gris)
cv2.waitKey(0)

-1

In [854]:
alpha = 1.3   # contraste
beta = 20     # brillo

ajustada = cv2.convertScaleAbs(gris, alpha=alpha, beta=beta)

cv2.imshow("Brillo y Contraste", ajustada)
cv2.waitKey(0)

-1

In [855]:
gamma = 1.5
invGamma = 1.0 / gamma

tabla = np.array([(i/255.0)**invGamma * 255
                  for i in np.arange(0,256)]).astype("uint8")

gamma_img = cv2.LUT(ajustada, tabla)

cv2.imshow("Correccion Gamma", gamma_img)
cv2.waitKey(0)

-1

In [856]:
ecualizada = cv2.equalizeHist(gamma_img)

cv2.imshow("Ecualizada", ecualizada)
cv2.waitKey(0)

-1

In [857]:
_, imagen_binaria = cv2.threshold(
    ecualizada,
    0,
    255,
    cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
)

cv2.imshow("Imagen Binaria", imagen_binaria)
cv2.waitKey(0)

-1

In [858]:
kernel = np.ones((5,5), np.uint8)

apertura = cv2.morphologyEx(imagen_binaria, cv2.MORPH_OPEN, kernel, iterations=2)
cierre = cv2.morphologyEx(apertura, cv2.MORPH_CLOSE, kernel, iterations=2)

cv2.imshow("Imagen Morfologica", cierre)
cv2.waitKey(0)

-1

In [859]:
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
    cierre,
    8,
    ltype=cv2.CV_32S
)

print("Numero de etiquetas:", num_labels)

Numero de etiquetas: 9


In [860]:
contador_dados = 0
dados = []

for i in range(1, num_labels):

    x = stats[i, cv2.CC_STAT_LEFT]
    y = stats[i, cv2.CC_STAT_TOP]
    w = stats[i, cv2.CC_STAT_WIDTH]
    h = stats[i, cv2.CC_STAT_HEIGHT]
    area = stats[i, cv2.CC_STAT_AREA]

    relacion = w / float(h)

   
    if area > 1000 and 0.7 < relacion < 1.3:
        contador_dados += 1
        dados.append((x,y,w,h))

        cv2.rectangle(imagen, (x,y), (x+w,y+h), (0,255,0), 2)

print("Numero de dados detectados:", contador_dados)

Numero de dados detectados: 2


In [861]:
for i, (x,y,w,h) in enumerate(dados):

    roi = imagen[y:y+h, x:x+w]

    gris_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)

    # Suavizado (clave para Hough)
    gris_roi = cv2.GaussianBlur(gris_roi, (9,9), 2)

  
    circulos = cv2.HoughCircles(
        gris_roi,
        cv2.HOUGH_GRADIENT,
        dp=1.2,
        minDist=20,
        param1=50,
        param2=10,
        minRadius=4,
        maxRadius=30
    )

    puntos = 0

    if circulos is not None:
        circulos = np.uint16(np.around(circulos))
        puntos = len(circulos[0])

 
        for c in circulos[0]:
            cx, cy, r = c
            cv2.circle(roi, (cx, cy), r, (0,255,0), 2)

    print(f"Dado {i+1}: {puntos} puntos")

    cv2.putText(imagen, str(puntos), (x, y-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)

    cv2.imshow("ROI", roi)
    cv2.waitKey(0)

Dado 1: 7 puntos
Dado 2: 3 puntos


In [862]:
cv2.imshow("Resultado Final", imagen)
cv2.waitKey(0)
cv2.destroyAllWindows()